In [2]:
# Import the tools used below.
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

In [3]:
class TieBreakQuantileTransformer(QuantileTransformer):
    """QuantileTransformer fitted on slightly jittered data.

    The plain version sends every value equal to a column's min or max to +/-5.2,
    so 0/1 flags and mostly-zero counts sit at +/-5.2. Jitter spreads tied values
    over their share of the quantiles; transform() still sees the original values,
    so a tied value lands in the middle of its share (a flag with 10% ones: 0 -> -0.13, 1 -> +1.64).
    """
    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        rng = np.random.default_rng(0)
        if self.subsample is not None and len(values) > self.subsample:
            values = values[rng.choice(len(values), self.subsample, replace=False)]  # it would subsample anyway
        sd = np.nanstd(values, axis=0)
        sd[~(sd > 0)] = 1.0                                    # constant or all-missing columns
        values = values + rng.standard_normal(values.shape) * (1e-3 * sd)
        if hasattr(X, 'columns'):                              # keep names so transform() doesn't warn
            values = pd.DataFrame(values, columns=X.columns)
        return super().fit(values, y)


In [ ]:
# Load applications and installment payments.
app_train_df = pd.read_csv('data/application_train.csv')
app_test_df = pd.read_csv('data/application_test.csv')
installments_payments_df = pd.read_csv('data/installments_payments.csv')
# 365243 means "not employed" (mostly pensioners), not a real tenure.
for df in (app_train_df, app_test_df):
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

In [5]:
# Split application dataframes; retain customer IDs to connect their histories.
torch.manual_seed(42)
MAX_HISTORY = 100
X_df = app_train_df.drop(columns='TARGET')
y_df = app_train_df['TARGET']
x_tr, x_te, _, y_te = train_test_split(
    X_df, y_df, test_size=0.2, stratify=y_df, random_state=42,
)

In [6]:
# Scale application numbers (missing stays NaN for the model); encode categories for embeddings.
num_cols = x_tr.select_dtypes('number').columns.drop('SK_ID_CURR')
cat_cols = x_tr.select_dtypes(exclude='number').columns
# Child tables reuse num_pipeline; their row encoders need imputed values.
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', TieBreakQuantileTransformer(output_distribution='normal', random_state=42)),
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__', keep_empty_features=True)),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])
preprocess = ColumnTransformer([
    ('num', TieBreakQuantileTransformer(output_distribution='normal', random_state=42), num_cols),
    ('cat', cat_pipeline, cat_cols),
])
preprocess.fit(x_tr)
# Transform labeled and test applications with the same fitted pipeline.
all_app_df = pd.concat([X_df, app_test_df], ignore_index=True)
n_labeled = len(X_df)
x_app = torch.tensor(preprocess.transform(all_app_df), dtype=torch.float32)
app_cat_sizes = [len(c) + 1 for c in preprocess.named_transformers_['cat']['encoder'].categories_]


In [8]:
# Keep individual payment rows, with at most 100 recent rows per applicant.
payments = installments_payments_df
# Per-row signals that scaling each column separately would wash out.
payments['DAYS_LATE'] = payments['DAYS_ENTRY_PAYMENT'] - payments['DAYS_INSTALMENT']  # > 0: paid after the due date
payments['AMT_SHORT'] = payments['AMT_INSTALMENT'] - payments['AMT_PAYMENT']         # > 0: paid less than due
payments['PAY_RATIO'] = (payments['AMT_PAYMENT'] / payments['AMT_INSTALMENT']).replace([np.inf, -np.inf], np.nan)
payments['IS_LATE'] = (payments['DAYS_LATE'] > 0).astype(float)
history_cols = ['NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER',
                'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT',
                'DAYS_LATE', 'AMT_SHORT', 'PAY_RATIO', 'IS_LATE']
history = payments.loc[
    payments['SK_ID_CURR'].isin(all_app_df['SK_ID_CURR']),
    ['SK_ID_CURR'] + history_cols,
].copy()
history = history.sort_values(['DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT'], ascending=False)
history = history.groupby('SK_ID_CURR', sort=False).head(MAX_HISTORY).copy()

# Fit installment preprocessing using training applicants only.
history_preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', TieBreakQuantileTransformer(output_distribution='normal', random_state=42)),
])
history_preprocess.fit(history.loc[history['SK_ID_CURR'].isin(x_tr['SK_ID_CURR']), history_cols])
rows = torch.tensor(history_preprocess.transform(history[history_cols]), dtype=torch.float32)


In [9]:
# Pad histories and mark real rows so padding is ignored inside the model.
positions = pd.Series(np.arange(len(all_app_df)), index=all_app_df['SK_ID_CURR'])
customer = history['SK_ID_CURR'].map(positions).to_numpy(copy=True)
slot = history.groupby('SK_ID_CURR', sort=False).cumcount().to_numpy(copy=True)
x_history = torch.zeros((len(all_app_df), MAX_HISTORY, rows.shape[1]))
mask = torch.zeros((len(all_app_df), MAX_HISTORY), dtype=torch.bool)
x_history[customer, slot] = rows
mask[customer, slot] = True
y = torch.tensor(y_df.to_numpy(), dtype=torch.float32)

# Release payment tables and temporary arrays after packing.
del installments_payments_df, payments, history, rows, customer, slot


In [10]:
x_history.shape

torch.Size([356255, 100, 10])

---

## Bureau

In [ ]:
# Summarise each bureau credit's whole monthly history (no 100-row cap, no mixing of credits).
# The summary becomes extra columns on that credit's bureau row.
bureau_balance_df = pd.read_csv('data/bureau_balance.csv', dtype={'STATUS': 'category'})
dpd_level = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}  # 'C' (closed) and 'X' (unknown) -> NaN
bureau_balance_df['DPD_LEVEL'] = bureau_balance_df['STATUS'].map(dpd_level).astype(float)
bureau_balance_df['IS_DPD'] = (bureau_balance_df['DPD_LEVEL'] > 0).astype(float)
by_credit = bureau_balance_df.groupby('SK_ID_BUREAU')
bureau_balance_summary = by_credit.agg(
    BB_MONTHS=('MONTHS_BALANCE', 'size'),
    BB_DPD_SHARE=('IS_DPD', 'mean'),
    BB_WORST_DPD=('DPD_LEVEL', 'max'),
)
recent = bureau_balance_df['MONTHS_BALANCE'] >= -12
bureau_balance_summary['BB_DPD_LAST_12M'] = bureau_balance_df[recent].groupby('SK_ID_BUREAU')['IS_DPD'].sum()
latest = bureau_balance_df.loc[by_credit['MONTHS_BALANCE'].idxmax(), ['SK_ID_BUREAU', 'STATUS']]
bureau_balance_summary['BB_LAST_STATUS'] = latest.set_index('SK_ID_BUREAU')['STATUS'].astype(str)
del bureau_balance_df, by_credit, recent, latest


In [ ]:
bureau_df = pd.read_csv('data/bureau.csv')

# Attach each credit's monthly-history summary; BB_MONTHS = 0 means no monthly records.
bureau_df = bureau_df.join(bureau_balance_summary, on='SK_ID_BUREAU')
bureau_df[['BB_MONTHS', 'BB_DPD_LAST_12M']] = bureau_df[['BB_MONTHS', 'BB_DPD_LAST_12M']].fillna(0)
del bureau_balance_summary
# Per-row signals.
bureau_df['DEBT_RATIO'] = (bureau_df['AMT_CREDIT_SUM_DEBT'] / bureau_df['AMT_CREDIT_SUM']).replace([np.inf, -np.inf], np.nan)
bureau_df['IS_OVERDUE'] = (bureau_df['CREDIT_DAY_OVERDUE'] > 0).astype(float)

# Select features; IDs stay available for matching applicants later.
bureau_num_cols = bureau_df.select_dtypes('number').columns.drop(
    ['SK_ID_CURR', 'SK_ID_BUREAU']
)
bureau_cat_cols = bureau_df.select_dtypes(exclude='number').columns

bureau_preprocess = ColumnTransformer([
    ('num', num_pipeline, bureau_num_cols),
    ('cat', cat_pipeline, bureau_cat_cols),
])

# Fit only on bureau records belonging to training applicants.
bureau_train = bureau_df['SK_ID_CURR'].isin(x_tr['SK_ID_CURR'])
bureau_preprocess.fit(bureau_df.loc[bureau_train])

# Transform every bureau record using the fitted preprocessing.
bureau_rows = torch.tensor(
    bureau_preprocess.transform(bureau_df),
    dtype=torch.float32,
)

# Embedding sizes, including index 0 for unseen categories.
bureau_cat_sizes = [
    len(c) + 1
    for c in bureau_preprocess.named_transformers_['cat']['encoder'].categories_
]

In [13]:
# Find the applicant and slot for each bureau record.
bureau_customer = bureau_df['SK_ID_CURR'].map(positions).to_numpy(copy=True)
bureau_slot = bureau_df.groupby('SK_ID_CURR', sort=False).cumcount().to_numpy(copy=True)

# Allocate enough slots to keep every bureau record.
MAX_BUREAU = int(bureau_slot.max()) + 1
x_bureau = torch.zeros(
    (len(all_app_df), MAX_BUREAU, bureau_rows.shape[1])
)
bureau_mask = torch.zeros(
    (len(all_app_df), MAX_BUREAU), dtype=torch.bool
)

# Place records into their slots and mark them as real.
x_bureau[bureau_customer, bureau_slot] = bureau_rows
bureau_mask[bureau_customer, bureau_slot] = True

# Keep bureau_df for linking monthly balances; release temporary arrays.
del bureau_rows, bureau_customer, bureau_train


In [14]:
x_bureau.shape

torch.Size([356255, 116, 22])

------------------------

## Bureau Balance

In [ ]:
# Link monthly records to applicants through their bureau credit IDs.
bureau_balance_df = pd.read_csv('data/bureau_balance.csv').merge(
    bureau_df[['SK_ID_BUREAU', 'SK_ID_CURR']], on='SK_ID_BUREAU', how='inner',
)
del bureau_df  # Applicant IDs are now attached to the monthly records.

# Keep the 100 most recent monthly records across each applicant's credits.
MAX_BUREAU_BALANCE = 100
bureau_balance_df = bureau_balance_df.sort_values('MONTHS_BALANCE', ascending=False)
bureau_balance_df = bureau_balance_df.groupby('SK_ID_CURR', sort=False).head(MAX_BUREAU_BALANCE)

# Scale months and encode status, fitting only on training applicants.
bureau_balance_num_cols = ['MONTHS_BALANCE']
bureau_balance_cat_cols = ['STATUS']
bureau_balance_preprocess = ColumnTransformer([
    ('num', num_pipeline, bureau_balance_num_cols),
    ('cat', cat_pipeline, bureau_balance_cat_cols),
])
bureau_balance_train = bureau_balance_df['SK_ID_CURR'].isin(x_tr['SK_ID_CURR'])
bureau_balance_preprocess.fit(bureau_balance_df.loc[bureau_balance_train])
bureau_balance_rows = torch.tensor(
    bureau_balance_preprocess.transform(bureau_balance_df), dtype=torch.float32,
)
bureau_balance_cat_sizes = [
    len(c) + 1
    for c in bureau_balance_preprocess.named_transformers_['cat']['encoder'].categories_
]

In [16]:
# Place each monthly record in its applicant's history and mark real slots.
bureau_balance_customer = bureau_balance_df['SK_ID_CURR'].map(positions).to_numpy(copy=True)
bureau_balance_slot = bureau_balance_df.groupby('SK_ID_CURR', sort=False).cumcount().to_numpy(copy=True)
x_bureau_balance = torch.zeros(
    (len(all_app_df), MAX_BUREAU_BALANCE, bureau_balance_rows.shape[1])
)
bureau_balance_mask = torch.zeros(
    (len(all_app_df), MAX_BUREAU_BALANCE), dtype=torch.bool
)
x_bureau_balance[bureau_balance_customer, bureau_balance_slot] = bureau_balance_rows
bureau_balance_mask[bureau_balance_customer, bureau_balance_slot] = True

# Only the packed tensors are needed for training.
del bureau_balance_df, bureau_balance_rows, bureau_balance_train
del bureau_balance_customer, bureau_balance_slot


---

## Credit card balance

In [ ]:
credit_card_balance_df = pd.read_csv('data/credit_card_balance.csv')

credit_card_balance_df['UTILIZATION'] = (credit_card_balance_df['AMT_BALANCE'] / credit_card_balance_df['AMT_CREDIT_LIMIT_ACTUAL']).replace([np.inf, -np.inf], np.nan)
credit_card_balance_df['PAID_VS_MIN'] = (credit_card_balance_df['AMT_PAYMENT_CURRENT'] / credit_card_balance_df['AMT_INST_MIN_REGULARITY']).replace([np.inf, -np.inf], np.nan)
credit_card_balance_df['IS_DPD'] = (credit_card_balance_df['SK_DPD'] > 0).astype(float)


# Keep the 100 most recent monthly records across each applicant's cards.
MAX_CREDIT_CARD = 100
credit_card_balance_df = credit_card_balance_df.sort_values('MONTHS_BALANCE', ascending=False)
credit_card_balance_df = credit_card_balance_df.groupby('SK_ID_CURR', sort=False).head(MAX_CREDIT_CARD)

# Select features and fit preprocessing on training applicants only.
credit_card_num_cols = credit_card_balance_df.select_dtypes('number').columns.drop(
    ['SK_ID_CURR', 'SK_ID_PREV']
)
credit_card_cat_cols = credit_card_balance_df.select_dtypes(exclude='number').columns
credit_card_preprocess = ColumnTransformer([
    ('num', num_pipeline, credit_card_num_cols),
    ('cat', cat_pipeline, credit_card_cat_cols),
])
credit_card_train = credit_card_balance_df['SK_ID_CURR'].isin(x_tr['SK_ID_CURR'])
credit_card_preprocess.fit(credit_card_balance_df.loc[credit_card_train])
credit_card_rows = torch.tensor(
    credit_card_preprocess.transform(credit_card_balance_df), dtype=torch.float32,
)
credit_card_cat_sizes = [
    len(c) + 1
    for c in credit_card_preprocess.named_transformers_['cat']['encoder'].categories_
]

In [18]:
# Place monthly card records into applicant histories and mark real slots.
credit_card_customer = credit_card_balance_df['SK_ID_CURR'].map(positions).to_numpy(copy=True)
credit_card_slot = credit_card_balance_df.groupby('SK_ID_CURR', sort=False).cumcount().to_numpy(copy=True)
x_credit_card = torch.zeros(
    (len(all_app_df), MAX_CREDIT_CARD, credit_card_rows.shape[1])
)
credit_card_mask = torch.zeros(
    (len(all_app_df), MAX_CREDIT_CARD), dtype=torch.bool
)
x_credit_card[credit_card_customer, credit_card_slot] = credit_card_rows
credit_card_mask[credit_card_customer, credit_card_slot] = True

# Release card records and temporary arrays after packing.
del credit_card_balance_df, credit_card_rows, credit_card_train
del credit_card_customer, credit_card_slot


## POS / cash balance

In [ ]:
pos_cash_balance_df = pd.read_csv('data/POS_CASH_balance.csv')

pos_cash_balance_df['INSTALMENTS_LEFT_SHARE'] = (pos_cash_balance_df['CNT_INSTALMENT_FUTURE'] / pos_cash_balance_df['CNT_INSTALMENT']).replace([np.inf, -np.inf], np.nan)
pos_cash_balance_df['IS_DPD'] = (pos_cash_balance_df['SK_DPD'] > 0).astype(float)

# Keep the 100 most recent monthly records across each applicant's POS/cash loans.
MAX_POS_CASH = 100
pos_cash_balance_df = pos_cash_balance_df.sort_values('MONTHS_BALANCE', ascending=False)
pos_cash_balance_df = pos_cash_balance_df.groupby('SK_ID_CURR', sort=False).head(MAX_POS_CASH)

# Scale numbers and encode categories, fitting only on training applicants.
pos_cash_num_cols = pos_cash_balance_df.select_dtypes('number').columns.drop(
    ['SK_ID_CURR', 'SK_ID_PREV']
)
pos_cash_cat_cols = pos_cash_balance_df.select_dtypes(exclude='number').columns
pos_cash_preprocess = ColumnTransformer([
    ('num', num_pipeline, pos_cash_num_cols),
    ('cat', cat_pipeline, pos_cash_cat_cols),
])
pos_cash_train = pos_cash_balance_df['SK_ID_CURR'].isin(x_tr['SK_ID_CURR'])
pos_cash_preprocess.fit(pos_cash_balance_df.loc[pos_cash_train])
pos_cash_rows = torch.tensor(
    pos_cash_preprocess.transform(pos_cash_balance_df), dtype=torch.float32,
)
pos_cash_cat_sizes = [
    len(c) + 1
    for c in pos_cash_preprocess.named_transformers_['cat']['encoder'].categories_
]


In [20]:
# Place monthly POS/cash records into applicant histories and mark real slots.
pos_cash_customer = pos_cash_balance_df['SK_ID_CURR'].map(positions).to_numpy(copy=True)
pos_cash_slot = pos_cash_balance_df.groupby('SK_ID_CURR', sort=False).cumcount().to_numpy(copy=True)
x_pos_cash = torch.zeros(
    (len(all_app_df), MAX_POS_CASH, pos_cash_rows.shape[1])
)
pos_cash_mask = torch.zeros(
    (len(all_app_df), MAX_POS_CASH), dtype=torch.bool
)
x_pos_cash[pos_cash_customer, pos_cash_slot] = pos_cash_rows
pos_cash_mask[pos_cash_customer, pos_cash_slot] = True

# Release raw records and temporary arrays after packing.
del pos_cash_balance_df, pos_cash_rows, pos_cash_train
del pos_cash_customer, pos_cash_slot


## Previous applications

In [ ]:
previous_application_df = pd.read_csv('data/previous_application.csv')

previous_application_df['CREDIT_TO_APPLICATION'] = (previous_application_df['AMT_CREDIT'] / previous_application_df['AMT_APPLICATION']).replace([np.inf, -np.inf], np.nan)
previous_application_df['ANNUITY_TO_CREDIT'] = (previous_application_df['AMT_ANNUITY'] / previous_application_df['AMT_CREDIT']).replace([np.inf, -np.inf], np.nan)
# Keep the 32 most recent applications per applicant to limit padding.
MAX_PREVIOUS = 32
previous_application_df = previous_application_df.sort_values('DAYS_DECISION', ascending=False)
previous_application_df = previous_application_df.groupby('SK_ID_CURR', sort=False).head(MAX_PREVIOUS)

# Mark missing-date codes as NaN; the pipeline will impute them.
previous_date_cols = previous_application_df.filter(like='DAYS_').columns
previous_application_df[previous_date_cols] = previous_application_df[previous_date_cols].replace(365243, np.nan)

# Exclude IDs from the model's features.
previous_num_cols = previous_application_df.select_dtypes('number').columns.drop(
    ['SK_ID_CURR', 'SK_ID_PREV']
)
previous_cat_cols = previous_application_df.select_dtypes(exclude='number').columns
previous_preprocess = ColumnTransformer([
    ('num', num_pipeline, previous_num_cols),
    ('cat', cat_pipeline, previous_cat_cols),
])

# Fit using training applicants and apply the same transform to every record.
previous_train = previous_application_df['SK_ID_CURR'].isin(x_tr['SK_ID_CURR'])
previous_preprocess.fit(previous_application_df.loc[previous_train])
previous_rows = torch.tensor(
    previous_preprocess.transform(previous_application_df), dtype=torch.float32,
)
previous_cat_sizes = [
    len(c) + 1
    for c in previous_preprocess.named_transformers_['cat']['encoder'].categories_
]


In [22]:
# Place previous applications into applicant histories and mark real slots.
previous_customer = previous_application_df['SK_ID_CURR'].map(positions).to_numpy(copy=True)
previous_slot = previous_application_df.groupby('SK_ID_CURR', sort=False).cumcount().to_numpy(copy=True)
x_previous = torch.zeros(
    (len(all_app_df), MAX_PREVIOUS, previous_rows.shape[1])
)
previous_mask = torch.zeros(
    (len(all_app_df), MAX_PREVIOUS), dtype=torch.bool
)
x_previous[previous_customer, previous_slot] = previous_rows
previous_mask[previous_customer, previous_slot] = True

# Release raw applications and temporary arrays after packing.
del previous_application_df, previous_rows, previous_train
del previous_customer, previous_slot


## Dataset and Dataloader

In [23]:
# Keep one copy of the inputs on CPU; loaders return only applicant indices.
input_tensors = (
    x_app, x_history, mask, x_bureau, bureau_mask,
    x_bureau_balance, bureau_balance_mask, x_credit_card, credit_card_mask,
    x_pos_cash, pos_cash_mask, x_previous, previous_mask
)
train_idx = x_tr.index.to_numpy(copy=True)
val_idx = x_te.index.to_numpy(copy=True)
test_idx = np.arange(n_labeled, len(all_app_df))

train_loader = DataLoader(train_idx, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_idx, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_idx, batch_size=1024, shuffle=False)


---

In [ ]:
# Encode applications and individual payments separately, then combine them.
class TabularModel(nn.Module):
    def __init__(self, n_num, app_cat_sizes, n_history, token_dim,
                 n_bureau_num, bureau_cat_sizes, n_bureau_balance_num, bureau_balance_cat_sizes,
                 n_credit_card_num, credit_card_cat_sizes,
                 n_pos_cash_num, pos_cash_cat_sizes, n_previous_num, previous_cat_sizes,
                 ):
        super().__init__()
        
        self.n_num = n_num

        self.embeddings = nn.ModuleList([nn.Embedding(size, token_dim) for size in app_cat_sizes])

        self.num_weight = nn.Parameter(torch.randn(n_num, token_dim) * 0.02)
        self.num_bias = nn.Parameter(torch.randn(n_num, token_dim) * 0.02)
        self.num_missing = nn.Parameter(torch.randn(n_num, token_dim) * 0.02)
        self.cls = nn.Parameter(torch.randn(1, 1, token_dim) * 0.02)


        #--------- bureau
        self.n_bureau_num = n_bureau_num
        self.bureau_embeddings = nn.ModuleList([
            nn.Embedding(size, token_dim) for size in bureau_cat_sizes
        ])
        self.bureau_encoder = nn.Sequential(
            nn.Linear(n_bureau_num + token_dim * len(bureau_cat_sizes), 64),
            nn.ReLU(),
            nn.Linear(64, token_dim),
            nn.ReLU(),
        )

        # Encode monthly credit balances into the same token width.
        self.n_bureau_balance_num = n_bureau_balance_num
        self.bureau_balance_embeddings = nn.ModuleList([
            nn.Embedding(size, token_dim) for size in bureau_balance_cat_sizes
        ])
        self.bureau_balance_encoder = nn.Sequential(
            nn.Linear(n_bureau_balance_num + token_dim * len(bureau_balance_cat_sizes), 64),
            nn.ReLU(),
            nn.Linear(64, token_dim), 
            nn.ReLU(),
        )

        # Encode each monthly card record into the common token width.
        self.n_credit_card_num = n_credit_card_num
        self.credit_card_embeddings = nn.ModuleList([
            nn.Embedding(size, token_dim) for size in credit_card_cat_sizes
        ])
        self.credit_card_encoder = nn.Sequential(
            nn.Linear(n_credit_card_num + token_dim * len(credit_card_cat_sizes), 64),
            nn.ReLU(), 
            nn.Linear(64, token_dim), 
            nn.ReLU(),
        )

        # Encode each POS/cash monthly record into the common token width.
        self.n_pos_cash_num = n_pos_cash_num
        self.pos_cash_embeddings = nn.ModuleList([
            nn.Embedding(size, token_dim) for size in pos_cash_cat_sizes
        ])
        self.pos_cash_encoder = nn.Sequential(
            nn.Linear(n_pos_cash_num + token_dim * len(pos_cash_cat_sizes), 64),
            nn.ReLU(), 
            nn.Linear(64, token_dim), 
            nn.ReLU(),
        )

        # Encode numeric values and categories of each previous application.
        self.n_previous_num = n_previous_num
        self.previous_embeddings = nn.ModuleList([
            nn.Embedding(size, token_dim) for size in previous_cat_sizes
        ])
        self.previous_encoder = nn.Sequential(
            nn.Linear(n_previous_num + token_dim * len(previous_cat_sizes), 64),
            nn.ReLU(),
            nn.Linear(64, token_dim), 
            nn.ReLU(),
        )

        #---------
        self.installments = nn.Sequential(
            nn.Linear(n_history, 64),
            nn.ReLU(),
            nn.Linear(64, token_dim),
            nn.ReLU(),
        )

        self.attention = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=token_dim, nhead=4, dim_feedforward=192, dropout=0.1,
                batch_first=True, norm_first=True,
            ),
            num_layers=3, norm=nn.LayerNorm(token_dim), enable_nested_tensor=False,
        )

        self.output = nn.Sequential(nn.Linear(token_dim, 8), nn.ReLU(), nn.Linear(8, 1))

    def forward(self, app, history, mask, bureau, bureau_mask,
                bureau_balance, bureau_balance_mask, credit_card, credit_card_mask,
                pos_cash, pos_cash_mask, previous, previous_mask):
        categories = app[:, self.n_num:].long() + 1  # Index 0 handles unseen categories.
        embeddings_list = [emb(categories[:, i]) for i, emb in enumerate(self.embeddings)]
        cat_tokens = torch.stack(embeddings_list, dim=1)

        numbers = app[:, :self.n_num].unsqueeze(-1)
        num_tokens = torch.where(
            numbers.isnan(), self.num_missing, numbers.nan_to_num() * self.num_weight + self.num_bias,
        )


        # Summarize the encoded payments into one installment token.
        rows = self.installments(history)
        mask = mask.unsqueeze(-1).to(rows.dtype)
        pooled = (rows * mask).sum(1) / mask.sum(1).clamp_min(1)


        # Embed the categorical columns of each bureau record.
        bureau_categories = bureau[:, :, self.n_bureau_num:].long() + 1
        bureau_embeddings = torch.cat([
            emb(bureau_categories[:, :, i])
            for i, emb in enumerate(self.bureau_embeddings)
        ], dim=-1)

        # Combine numeric values and embeddings, then encode each record.
        bureau_features = torch.cat([
            bureau[:, :, :self.n_bureau_num], bureau_embeddings
        ], dim=-1)
        bureau_encoded = self.bureau_encoder(bureau_features)

        # Pool real records into one bureau token.
        bureau_mask = bureau_mask.unsqueeze(-1).to(bureau_encoded.dtype)
        bureau_pooled = (
            (bureau_encoded * bureau_mask).sum(1)
            / bureau_mask.sum(1).clamp_min(1)
        )

        
        # Embed status and encode each monthly record before pooling.
        balance_categories = bureau_balance[:, :, self.n_bureau_balance_num:].long() + 1
        balance_embeddings = torch.cat([
            emb(balance_categories[:, :, i])
            for i, emb in enumerate(self.bureau_balance_embeddings)
        ], dim=-1)
        balance_features = torch.cat([
            bureau_balance[:, :, :self.n_bureau_balance_num], balance_embeddings
        ], dim=-1)
        balance_encoded = self.bureau_balance_encoder(balance_features)
        bureau_balance_mask = bureau_balance_mask.unsqueeze(-1).to(balance_encoded.dtype)
        balance_pooled = (
            (balance_encoded * bureau_balance_mask).sum(1)
            / bureau_balance_mask.sum(1).clamp_min(1)
        )

        # Embed contract status and summarize real monthly card records.
        credit_card_categories = credit_card[:, :, self.n_credit_card_num:].long() + 1
        credit_card_embeddings = torch.cat([
            emb(credit_card_categories[:, :, i])
            for i, emb in enumerate(self.credit_card_embeddings)
        ], dim=-1)
        credit_card_features = torch.cat([
            credit_card[:, :, :self.n_credit_card_num], credit_card_embeddings
        ], dim=-1)
        credit_card_encoded = self.credit_card_encoder(credit_card_features)
        credit_card_mask = credit_card_mask.unsqueeze(-1).to(credit_card_encoded.dtype)
        credit_card_pooled = (
            (credit_card_encoded * credit_card_mask).sum(1)
            / credit_card_mask.sum(1).clamp_min(1)
        )

        # Summarize real POS/cash monthly records into one token.
        pos_cash_categories = pos_cash[:, :, self.n_pos_cash_num:].long() + 1
        pos_cash_embeddings = torch.cat([
            emb(pos_cash_categories[:, :, i])
            for i, emb in enumerate(self.pos_cash_embeddings)
        ], dim=-1)
        pos_cash_features = torch.cat([
            pos_cash[:, :, :self.n_pos_cash_num], pos_cash_embeddings
        ], dim=-1)
        pos_cash_encoded = self.pos_cash_encoder(pos_cash_features)
        pos_cash_mask = pos_cash_mask.unsqueeze(-1).to(pos_cash_encoded.dtype)
        pos_cash_pooled = (
            (pos_cash_encoded * pos_cash_mask).sum(1)
            / pos_cash_mask.sum(1).clamp_min(1)
        )

        # Summarize encoded previous applications into one token.
        previous_categories = previous[:, :, self.n_previous_num:].long() + 1
        previous_embeddings = torch.cat([
            emb(previous_categories[:, :, i])
            for i, emb in enumerate(self.previous_embeddings)
        ], dim=-1)
        previous_features = torch.cat([
            previous[:, :, :self.n_previous_num], previous_embeddings
        ], dim=-1)
        previous_encoded = self.previous_encoder(previous_features)
        previous_mask = previous_mask.unsqueeze(-1).to(previous_encoded.dtype)
        previous_pooled = (
            (previous_encoded * previous_mask).sum(1)
            / previous_mask.sum(1).clamp_min(1)
        )


        # CLS goes first; the prediction is read from it after attention.
        tokens = torch.cat([
            self.cls.expand(app.shape[0], -1, -1),
            num_tokens, cat_tokens, bureau_pooled.unsqueeze(1),
            balance_pooled.unsqueeze(1), credit_card_pooled.unsqueeze(1),
            pos_cash_pooled.unsqueeze(1), previous_pooled.unsqueeze(1), pooled.unsqueeze(1)
        ], dim=1)

        
        attended = self.attention(tokens)

        return self.output(attended[:, 0]).squeeze(1)


In [30]:
SEEDS = [0, 1, 2, 3, 4]
criterion = nn.BCEWithLogitsLoss()

def predict(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for idx in loader:
            inputs = [tensor[idx].to('cuda') for tensor in input_tensors]
            out.append(model(*inputs).sigmoid().cpu().numpy())
    return np.concatenate(out)

val_preds, test_preds, seed_aucs = [], [], []
for seed in SEEDS:
    torch.manual_seed(seed)                                                     # init + dropout
    train_loader = DataLoader(train_idx, batch_size=1024, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))   # batch order
    model = TabularModel(
        n_num=x_app.shape[1] - len(cat_cols),
        app_cat_sizes=app_cat_sizes,
        n_history=x_history.shape[2],
        token_dim=64,
        n_bureau_num=len(bureau_num_cols),
        bureau_cat_sizes=bureau_cat_sizes,
        n_bureau_balance_num=len(bureau_balance_num_cols),
        bureau_balance_cat_sizes=bureau_balance_cat_sizes,
        n_credit_card_num=len(credit_card_num_cols),
        credit_card_cat_sizes=credit_card_cat_sizes,
        n_pos_cash_num=len(pos_cash_num_cols),
        pos_cash_cat_sizes=pos_cash_cat_sizes,
        n_previous_num=len(previous_num_cols),
        previous_cat_sizes=previous_cat_sizes,
    ).to('cuda')
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    best_auc, best_epoch, best_state = 0, 0, None                               # reset every seed

    for epoch in tqdm(range(200), desc=f'seed {seed}'):
        model.train()
        for idx in train_loader:
            inputs = [tensor[idx].to('cuda') for tensor in input_tensors]
            target = y[idx].to('cuda')
            optimizer.zero_grad()
            loss = criterion(model(*inputs), target)
            loss.backward()
            optimizer.step()
        auc = roc_auc_score(y_te, predict(model, val_loader))
        if auc > best_auc:
            best_auc, best_epoch = auc, epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        elif epoch - best_epoch == 5:
            break

    model.load_state_dict(best_state)
    val_preds.append(predict(model, val_loader))
    test_preds.append(predict(model, test_loader))
    seed_aucs.append(best_auc)
    print(f'seed {seed}: best epoch {best_epoch + 1}, AUC = {best_auc:.4f}')

    torch.save({
        "state_dict": {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
        },
        "seed": int(seed),
        "best_epoch": int(best_epoch + 1),
        "best_auc": float(best_auc),
        "model_kwargs": {
            "n_num": x_app.shape[1] - len(cat_cols),
            "app_cat_sizes": app_cat_sizes,
            "n_history": x_history.shape[2],
            "token_dim": 64,
            "n_bureau_num": len(bureau_num_cols),
            "bureau_cat_sizes": bureau_cat_sizes,
            "n_bureau_balance_num": len(bureau_balance_num_cols),
            "bureau_balance_cat_sizes": bureau_balance_cat_sizes,
            "n_credit_card_num": len(credit_card_num_cols),
            "credit_card_cat_sizes": credit_card_cat_sizes,
            "n_pos_cash_num": len(pos_cash_num_cols),
            "pos_cash_cat_sizes": pos_cash_cat_sizes,
            "n_previous_num": len(previous_num_cols),
            "previous_cat_sizes": previous_cat_sizes,
        },
    }, f"/workspace/best_seed_{seed}.pt")

print(f'single seeds:   {np.mean(seed_aucs):.4f} ± {np.std(seed_aucs):.4f}')
print(f'5-seed average: {roc_auc_score(y_te, np.mean(val_preds, axis=0)):.4f}')




seed 0:   0%|          | 0/200 [00:00<?, ?it/s]

seed 0:   9%|▉         | 18/200 [05:13<52:54, 17.44s/it] 


seed 0: best epoch 14, AUC = 0.7884


seed 1:   8%|▊         | 15/200 [09:33<1:57:56, 38.25s/it]


seed 1: best epoch 11, AUC = 0.7859


seed 2:   8%|▊         | 17/200 [05:14<56:23, 18.49s/it]  


seed 2: best epoch 13, AUC = 0.7861


seed 3:   8%|▊         | 17/200 [04:34<49:12, 16.14s/it]


seed 3: best epoch 13, AUC = 0.7850


seed 4:   8%|▊         | 15/200 [04:03<50:08, 16.26s/it]


seed 4: best epoch 11, AUC = 0.7844
single seeds:   0.7860 ± 0.0014
5-seed average: 0.7905


In [ ]:
submission = pd.DataFrame({
    'SK_ID_CURR': app_test_df['SK_ID_CURR'].to_numpy(),
    'TARGET': np.mean(test_preds, axis=0),
})
submission.to_csv('./nn_comp_submission.csv', index=False)